# Week 2 — Bulk Parsing Pipeline

**Goal:** Parse all personal replays, apply filters, and produce a clean DataFrame ready for feature engineering.

**Filters applied:**
- Rated game (`de['rated'] == True`)
- Arabia map (`rms_map_id == 9`)
- 1v1 (`num_players == 2`)
- Duration > 8 minutes (removes disconnects and early resigns)

**Output:** `data/parsed_replays.csv` — one row per player per game (2 rows per 1v1)

## Setup

In [18]:
import struct
import logging
import pandas as pd
from pathlib import Path
from mgz.fast import header as fast_header, operation, Operation, Action, meta

# Constants
ARABIA_MAP_ID = 9
AGE_UP_TECH_IDS = {101: 'feudal', 102: 'castle', 103: 'imperial'}
RM_1V1_LEADERBOARD_ID = 3
MIN_DURATION_MIN = 8

REPLAY_DIR = Path('C:/Users/liher/Games/Age of Empires 2 DE/76561198151543542/savegame')
OUTPUT_PATH = Path('../data/parsed_replays.csv')

# Logging — failures written to file, not just printed
logging.basicConfig(
    filename='../data/parsing_failures.log',
    level=logging.ERROR,
    format='%(asctime)s — %(message)s'
)

print('Imports OK')

Imports OK


## Section 1 — Find Replay Files

In [19]:
replay_files = sorted(REPLAY_DIR.glob('*.aoe2record'))
print(f'Found {len(replay_files)} total replay files')

Found 3627 total replay files


## Section 2 — Parse All Replays

Runs `parse_replay()` on every file. Applies all four filters before keeping a result.
Parse failures are logged to `data/parsing_failures.log` and do not stop the batch.

In [20]:
def parse_header(f):
    """Parse header only. File cursor is left at start of body — ready for meta(f) + operation loop."""
    h = fast_header.parse(f)
    de = h.get('de') or {}
    header_players = {p['number']: p for p in (h.get('players') or []) if p and p['type'] == 1}
    de_players = {p['number']: p for p in (de.get('players') or []) if p.get('number', -1) >= 1}
    map_id = de.get('rms_map_id')
    rated = de.get('rated', False)
    num_players = len(header_players)
    return header_players, de_players, map_id, rated, num_players


def parse_body(f):
    """Scan body operations. File cursor must be positioned after header."""
    meta(f)

    game_time_ms = 0
    age_up_times = {}
    resign_player = None
    resign_time_min = None
    leaderboard_ratings = {}
    world_time_ms = None

    try:
        while True:
            op_type, payload = operation(f)

            if op_type == Operation.SYNC:
                increment, _, _ = payload
                game_time_ms += increment

            elif op_type == Operation.ACTION:
                action_type, ap = payload
                t_min = game_time_ms / 60000

                if action_type == Action.RESIGN:
                    pid = ap.get('player_id')
                    if resign_player is None:
                        resign_player = pid
                        resign_time_min = round(t_min, 2)

                elif action_type == Action.RESEARCH:
                    tech = ap.get('technology_id')
                    pid = ap.get('player_id')
                    if tech in AGE_UP_TECH_IDS:
                        age = AGE_UP_TECH_IDS[tech]
                        age_up_times.setdefault(pid, {})
                        if age not in age_up_times[pid]:
                            age_up_times[pid][age] = round(t_min, 2)

            elif op_type == Operation.POSTGAME:
                world_time_ms = payload.get('world_time')
                for lb in (payload.get('leaderboards') or []):
                    if lb.get('id') == RM_1V1_LEADERBOARD_ID:
                        for entry in lb.get('players') or []:
                            leaderboard_ratings[entry['number']] = entry['rating']
                break
    except (EOFError, struct.error):
        pass

    duration_min = round((world_time_ms or game_time_ms) / 60000, 2)
    return age_up_times, resign_player, resign_time_min, leaderboard_ratings, duration_min

In [21]:
# Calculating run time and parsing times for header only
import time
sample = replay_files[:20]
times = []
for fp in sample:
    t = time.time()
    try:
        with open(fp, 'rb') as f:
            parse_header(f)
    except:
        pass
    times.append(time.time() - t)

print(f'Header parse avg: {sum(times)/len(times):.2f}s')
print(f'Header parse max: {max(times):.2f}s')

Header parse avg: 0.05s
Header parse max: 0.27s


In [22]:
import os
print(os.cpu_count())


12


In [23]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Phase 1: Single-threaded header scan ─────────────────────────────────────
# Loop all 3627 files cheaply. Only reads the compressed header block —
# no body scan yet. Filter by map, rated, and player count.
# Files that fail any filter are skipped entirely — body scan never happens.
print('Phase 1 — header scan...')
candidates = []
skipped_header = 0
header_errors = []
t1 = time.time()

for filepath in replay_files:
    try:
        with open(filepath, 'rb') as f:
            header_players, de_players, map_id, rated, num_players = parse_header(f)

        if map_id == ARABIA_MAP_ID and rated and num_players == 2:
            candidates.append((str(filepath), de_players))
        else:
            skipped_header += 1

    except Exception as e:
        header_errors.append(filepath.name)
        logging.error(f'HEADER FAILED: {filepath} | {type(e).__name__}: {e}')

print(f'  Done in {round(time.time()-t1, 1)}s')
print(f'  Candidates: {len(candidates)}  |  Skipped: {skipped_header}  |  Errors: {len(header_errors)}')


# ── Phase 2: Multithreaded body scan ─────────────────────────────────────────
# Take only the ~215 candidates that passed the header filter.
# Split across 12 threads — each thread independently opens its assigned files
# and scans the body. File I/O and zlib decompression release Python's GIL,
# so threads run in parallel for those operations giving real speedup.
# ThreadPoolExecutor works in Jupyter on Windows (unlike ProcessPoolExecutor).
def scan_body(args):
    """Worker function — runs in a thread. Re-opens the file, re-parses the
    header to reposition the cursor to the body start, then scans the body."""
    filepath_str, de_players = args
    try:
        with open(filepath_str, 'rb') as f:
            fast_header.parse(f)  # reposition cursor to body start
            age_up_times, resign_player, resign_time_min, leaderboard_ratings, duration_min = parse_body(f)

        return {
            'filepath':            filepath_str,
            'duration_min':        duration_min,
            'resign_player':       resign_player,
            'resign_time_min':     resign_time_min,
            'age_up_times':        age_up_times,
            'leaderboard_ratings': leaderboard_ratings,
            'de_players':          de_players,
            'error':               None,
        }
    except Exception as e:
        return {'filepath': filepath_str, 'error': str(e)}


print(f'\nPhase 2 — body scan ({len(candidates)} files across 12 threads)...')
t2 = time.time()
body_results = []
body_errors = []

with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(scan_body, args): args[0] for args in candidates}
    done = 0
    for future in as_completed(futures):
        result = future.result()
        done += 1
        if result.get('error'):
            body_errors.append(result['filepath'])
        else:
            body_results.append(result)
        if done % 25 == 0:
            print(f'  {done}/{len(candidates)} complete')

print(f'  Done in {round(time.time()-t2, 1)}s')
print(f'  Parsed: {len(body_results)}  |  Errors: {len(body_errors)}')


# ── Phase 3: Duration filter ──────────────────────────────────────────────────
# Drop games under 8 minutes (disconnects, early resigns on spawn).
# Duration can only be checked after body scan — requires SYNC increments
# or POSTGAME world_time, both of which are in the body.
print(f'\nPhase 3 — duration filter...')
results = []
skipped_duration = 0

for r in body_results:
    if r['duration_min'] <= MIN_DURATION_MIN:
        skipped_duration += 1
        continue
    results.append(r)

elapsed = round(time.time() - t1, 1)
print(f'\nDone in {elapsed}s total')
print(f'  Total files:           {len(replay_files)}')
print(f'  Kept:                  {len(results)}')
print(f'  Skipped (header):      {skipped_header}  (wrong map / unrated / team game)')
print(f'  Skipped (duration):    {skipped_duration}  (under {MIN_DURATION_MIN} min)')
print(f'  Parse errors (header): {len(header_errors)}')
print(f'  Parse errors (body):   {len(body_errors)}')

Phase 1 — header scan...
  Done in 1377.9s
  Candidates: 215  |  Skipped: 3407  |  Errors: 5

Phase 2 — body scan (215 files across 12 threads)...
  25/215 complete
  50/215 complete
  75/215 complete
  100/215 complete
  125/215 complete
  150/215 complete
  175/215 complete
  200/215 complete
  Done in 66.8s
  Parsed: 215  |  Errors: 0

Phase 3 — duration filter...

Done in 1444.7s total
  Total files:           3627
  Kept:                  194
  Skipped (header):      3407  (wrong map / unrated / team game)
  Skipped (duration):    21  (under 8 min)
  Parse errors (header): 5
  Parse errors (body):   0


In [24]:
df_raw = pd.DataFrame(results)
df_raw.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(df_raw)} rows to {OUTPUT_PATH}')

Saved 194 rows to ..\data\parsed_replays.csv
